In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [8]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


### Load Player Data and Bookmaker Data

In [15]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

C:\Users\alexg\AppData\Local\Temp\ipykernel_75696\1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Jalen Duren,Over,20.5,-137,2025-11-18,2025-11-17T23:35:53Z
1,Underdog,player_points,Jalen Duren,Under,20.5,-137,2025-11-18,2025-11-17T23:35:53Z
2,Underdog,player_points,Isaiah Stewart II,Over,11.5,-137,2025-11-18,2025-11-17T23:35:53Z
3,Underdog,player_points,Isaiah Stewart II,Under,11.5,-137,2025-11-18,2025-11-17T23:35:53Z
4,Underdog,player_points,Caris LeVert,Over,12.5,-137,2025-11-18,2025-11-17T23:35:53Z


### Update projected starting lineups

In [18]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\PRODUCTION/teamInfo.py
Updated 16 teams with confirmed lineups


### Top EVs for single bets

In [19]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
edge_threshold=0.30, stake=10, variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, 
max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'EXPECTED ROI', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 142 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,EXPECTED ROI,KELLY_FRACTION,SIGMA FLAG
0,Josh Giddey,Bovada,21.5,25.38,Over,200,0,10.56,105.6,0.528,High
1,Simone Fontecchio,Bovada,11.5,13.35,Over,205,0,9.28,92.8,0.453,High
2,Josh Giddey,Bovada,20.5,25.38,Over,160,1,8.93,89.3,0.558,High
3,Naji Marshall,Bovada,12.5,15.16,Over,180,0,8.42,84.2,0.468,High
4,LaMelo Ball,Bovada,23.5,25.13,Over,205,0,7.83,78.3,0.382,High


## Top EVs for 2 leg bets

### Underdog picks

In [20]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

Pre-computing predictions for 66 players...
Processing 60 players with valid predictions...
Generated 1669 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 89 combinations from 1669 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Nikola Jokić,Josh Giddey,27.5,18.5,22.11,25.38,under,over,1,7.53,0.376,High,High
1,Zion Williamson,Josh Giddey,17.5,18.5,22.72,25.38,over,over,1,7.46,0.373,High,High
2,Zion Williamson,Nikola Jokić,17.5,27.5,22.72,22.11,over,under,1,7.15,0.357,High,High
3,Chaz Lanier,Nikola Jokić,5.5,27.5,8.23,22.11,over,under,0,6.22,0.311,Med,High
4,Chaz Lanier,Josh Giddey,5.5,18.5,8.23,25.38,over,over,0,6.07,0.303,Med,High


### Prizepicks picks

In [21]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 108 players...
Processing 100 players with valid predictions...
Generated 4677 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 149 combinations from 4677 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Nikola Jokić,Josh Giddey,0.5,17.5,22.11,25.38,over,over,1,12.55,0.628,High,High
1,Simone Fontecchio,Nikola Jokić,8.5,0.5,13.35,22.11,over,over,1,11.61,0.581,High,High
2,Zion Williamson,Nikola Jokić,17.5,0.5,22.72,22.11,over,over,1,11.31,0.566,High,High
3,Simone Fontecchio,Josh Giddey,8.5,17.5,13.35,25.38,over,over,1,8.05,0.402,High,High
4,Zion Williamson,Josh Giddey,17.5,17.5,22.72,25.38,over,over,1,8.02,0.401,High,High


## 3 leg parlay

### Underdog picks

In [22]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 66 players...
Processing 60 players with valid predictions...
Generated 33704 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 39 combinations from 33704 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Zion Williamson,Nikola Jokić,Josh Giddey,17.5,27.5,18.5,22.72,22.11,25.38,over,under,over,1,14.87,0.297,High,High,High
1,Chaz Lanier,Nikola Jokić,Josh Giddey,5.5,27.5,18.5,8.23,22.11,25.38,over,under,over,0,13.22,0.264,Med,High,High
2,Chaz Lanier,Karl-Anthony Towns,Zion Williamson,5.5,28.5,17.5,8.23,24.27,22.72,over,under,over,0,10.21,0.204,Med,High,High
3,Karl-Anthony Towns,D'Angelo Russell,Alex Caruso,28.5,11.5,5.5,24.27,15.64,7.32,under,over,over,0,6.93,0.139,High,High,Med
4,D'Angelo Russell,P.J. Washington,Alex Caruso,11.5,15.5,5.5,15.64,19.53,7.32,over,over,over,0,6.60,0.132,High,High,Med


### Prizepicks picks

In [23]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 108 players...
Processing 100 players with valid predictions...
Generated 159630 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 66 combinations from 159630 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Simone Fontecchio,Nikola Jokić,Josh Giddey,8.5,0.5,17.5,13.35,22.11,25.38,over,over,over,1,22.52,0.450,High,High,High
1,Zion Williamson,Nikola Jokić,Josh Giddey,17.5,0.5,17.5,22.72,22.11,25.38,over,over,over,1,22.21,0.444,High,High,High
2,Simone Fontecchio,Naji Marshall,Zion Williamson,8.5,10.0,17.5,13.35,15.16,22.72,over,over,over,1,13.89,0.278,High,High,High
3,LaMelo Ball,Cooper Flagg,Naji Marshall,19.5,15.5,10.0,25.13,20.98,15.16,over,over,over,1,12.36,0.247,High,High,High
4,Jarace Walker,LaMelo Ball,Cooper Flagg,9.5,19.5,15.5,13.75,25.13,20.98,over,over,over,0,11.35,0.227,High,High,High
